In [1]:
!python -m pip install sqlalchemy

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd 
import numpy as np
import sqlite3
from sqlalchemy import create_engine
import os

In [3]:
engine = create_engine("sqlite:///../bluestock_mf.db")

In [4]:
conn = sqlite3.connect("../bluestock_mf.db")
cursor = conn.cursor()

with open("../sql/star_schema.sql", "r") as f:
    cursor.executescript(f.read())

conn.commit()

In [5]:
conn = sqlite3.connect("../bluestock_mf.db")

cursor = conn.cursor()

cursor.execute(
    "SELECT name FROM sqlite_master WHERE type='table';"
)

print(cursor.fetchall())

conn.close()

[('dim_fund',), ('sqlite_sequence',), ('dim_date',), ('fact_nav',), ('fact_transactions',), ('fact_performance',), ('fact_aum',)]


In [6]:
folder = os.path.join(os.path.dirname(os.getcwd()),'data','processed')

In [7]:
df = pd.read_csv(os.path.join(folder,'c_fund_master.csv'))

In [8]:
df.head()

,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code,exit_load_pct_anomaly,expense_ratio_pct_anomaly,min_sip_amount_anomaly,min_lumpsum_amount_anomaly
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01,False,False,False,False
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01,False,False,False,False
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03,False,False,False,False
3,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap,Direct,2013-01-01,BSE 250 SmallCap TRI,0.72,1.0,500,1000,R. Srinivasan,Very High,EC03,False,False,False,False
4,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Gilt,Regular,2000-12-30,CRISIL Dynamic Gilt Index,0.77,0.0,500,1000,Dinesh Ahuja,Low,DC02,True,False,False,False


In [9]:
df['launch_date'] = pd.to_datetime(df['launch_date'])

In [10]:
dim_fund = df[
    [
        "amfi_code",
        "fund_house",
        "scheme_name",
        "category",
        "sub_category",
        "plan",
        "launch_date",
        "benchmark",
        "fund_manager",
        "risk_category",
        "sebi_category_code"
    ]
].copy()

In [11]:
dim_fund.dtypes

amfi_code                      int64
fund_house                    object
scheme_name                   object
category                      object
sub_category                  object
plan                          object
launch_date           datetime64[ns]
benchmark                     object
fund_manager                  object
risk_category                 object
sebi_category_code            object
dtype: object

In [12]:
dim_fund.to_sql(
    "dim_fund",
    engine,
    if_exists="append",
    index=False
)

40

In [13]:
pd.read_sql(
    "SELECT * FROM dim_fund LIMIT 5;",
    engine
)

,fund_id,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,fund_manager,risk_category,sebi_category_code
0,1,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14 00:00:00.000000,NIFTY 100 TRI,Sohini Andani,Moderate,EC01
1,2,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01 00:00:00.000000,NIFTY 100 TRI,Sohini Andani,Moderate,EC01
2,3,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09 00:00:00.000000,BSE 250 SmallCap TRI,R. Srinivasan,Very High,EC03
3,4,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap,Direct,2013-01-01 00:00:00.000000,BSE 250 SmallCap TRI,R. Srinivasan,Very High,EC03
4,5,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Gilt,Regular,2000-12-30 00:00:00.000000,CRISIL Dynamic Gilt Index,Dinesh Ahuja,Low,DC02


In [24]:
import os
import glob
import pandas as pd


dfs = {}

for file in glob.glob(os.path.join(folder, "*.csv")):
    name = os.path.splitext(os.path.basename(file))[0] 
    dfs[name] = pd.read_csv(file)

print("Loaded files:")
print(list(dfs.keys()))

Loaded files:
['c_aum_by_fund_house', 'c_benchmark_indices', 'c_category_inflows', 'c_fund_master', 'c_industry_folio_count', 'c_investor_transaction', 'c_monthly_sip_inflows', 'c_nav_history', 'c_portfolio_holdings', 'c_scheme_performance']


In [25]:
fund_master = dfs["c_fund_master"]
aum = dfs["c_aum_by_fund_house"]
benchmark = dfs["c_benchmark_indices"]
category = dfs["c_category_inflows"]
industry_folio = dfs["c_industry_folio_count"]
investor_txn = dfs["c_investor_transaction"]
monthly_sip = dfs["c_monthly_sip_inflows"]
nav_history = dfs["c_nav_history"]
portfolio_holdings = dfs["c_portfolio_holdings"]
scheme_performance = dfs["c_scheme_performance"]

In [27]:
fund_master.dtypes

amfi_code                       int64
fund_house                     object
scheme_name                    object
category                       object
sub_category                   object
plan                           object
launch_date                    object
benchmark                      object
expense_ratio_pct             float64
exit_load_pct                 float64
min_sip_amount                  int64
min_lumpsum_amount              int64
fund_manager                   object
risk_category                  object
sebi_category_code             object
exit_load_pct_anomaly            bool
expense_ratio_pct_anomaly        bool
min_sip_amount_anomaly           bool
min_lumpsum_amount_anomaly       bool
dtype: object

In [22]:
folder

'C:\\Users\\dell\\OneDrive\\Desktop\\mutual-fund-capstone\\data\\processed'

In [23]:
fund_master.to_csv(os.path.join(folder,"c_fund_master.csv"),index=False)

In [33]:
df = pd.read_csv(os.path.join(folder,"c_fund_master.csv"))

In [35]:
df['launch_date'] = pd.to_datetime(df['launch_date'])

In [36]:
df.dtypes

amfi_code                              int64
fund_house                            object
scheme_name                           object
category                              object
sub_category                          object
plan                                  object
launch_date                   datetime64[ns]
benchmark                             object
expense_ratio_pct                    float64
exit_load_pct                        float64
min_sip_amount                         int64
min_lumpsum_amount                     int64
fund_manager                          object
risk_category                         object
sebi_category_code                    object
exit_load_pct_anomaly                   bool
expense_ratio_pct_anomaly               bool
min_sip_amount_anomaly                  bool
min_lumpsum_amount_anomaly              bool
dtype: object

In [37]:
df.to_csv(os.path.join(folder,'c_fund_master.csv'),index= False)

In [38]:
import os
import glob
import pandas as pd


dfs = {}

for file in glob.glob(os.path.join(folder, "*.csv")):
    name = os.path.splitext(os.path.basename(file))[0] 
    dfs[name] = pd.read_csv(file)

print("Loaded files:")
print(list(dfs.keys()))

Loaded files:
['c_aum_by_fund_house', 'c_benchmark_indices', 'c_category_inflows', 'c_fund_master', 'c_industry_folio_count', 'c_investor_transaction', 'c_monthly_sip_inflows', 'c_nav_history', 'c_portfolio_holdings', 'c_scheme_performance']


In [39]:
fund_master = dfs["c_fund_master"]
aum = dfs["c_aum_by_fund_house"]
benchmark = dfs["c_benchmark_indices"]
category = dfs["c_category_inflows"]
industry_folio = dfs["c_industry_folio_count"]
investor_txn = dfs["c_investor_transaction"]
monthly_sip = dfs["c_monthly_sip_inflows"]
nav_history = dfs["c_nav_history"]
portfolio_holdings = dfs["c_portfolio_holdings"]
scheme_performance = dfs["c_scheme_performance"]

In [84]:
dates = pd.concat([
    fund_master["launch_date"],
    nav_history["date"],
    aum["date"],
    monthly_sip["month"],
    category["month"],
    industry_folio["month"],
    investor_txn["transaction_date"],
    portfolio_holdings["portfolio_date"],
    benchmark["date"]
])

In [88]:
dates = dates.drop_duplicates()

In [90]:
dates = pd.to_datetime(dates)

In [93]:
dates = dates.sort_values()

In [94]:
dim_date = dates.to_frame(name="full_date")

In [95]:
dim_date

,full_date
5,1996-09-11
19,1999-08-19
31,2000-03-06
30,2000-04-17
4,2000-12-30
...,...
1603,2026-05-25
1604,2026-05-26
1605,2026-05-27
1606,2026-05-28


In [96]:
dim_date["day"] = dim_date["full_date"].dt.day
dim_date["month"] = dim_date["full_date"].dt.month
dim_date["month_name"] = dim_date["full_date"].dt.month_name()
dim_date["quarter"] = dim_date["full_date"].dt.quarter

In [97]:
dim_date["year"] = dim_date["full_date"].dt.year

In [98]:
dim_date

,full_date,day,month,month_name,quarter,year
5,1996-09-11,11,9,September,3,1996
19,1999-08-19,19,8,August,3,1999
31,2000-03-06,6,3,March,1,2000
30,2000-04-17,17,4,April,2,2000
4,2000-12-30,30,12,December,4,2000
...,...,...,...,...,...,...
1603,2026-05-25,25,5,May,2,2026
1604,2026-05-26,26,5,May,2,2026
1605,2026-05-27,27,5,May,2,2026
1606,2026-05-28,28,5,May,2,2026


In [99]:
dim_date.to_sql(
    "dim_date",
    engine,
    if_exists="append",
    index=False
)

1643

In [100]:
pd.read_sql(
    "SELECT * FROM dim_date LIMIT 10;",
    engine
)

,date_id,full_date,day,month,month_name,quarter,year
0,1,1996-09-11 00:00:00.000000,11,9,September,3,1996
1,2,1999-08-19 00:00:00.000000,19,8,August,3,1999
2,3,2000-03-06 00:00:00.000000,6,3,March,1,2000
3,4,2000-04-17 00:00:00.000000,17,4,April,2,2000
4,5,2000-12-30 00:00:00.000000,30,12,December,4,2000
5,6,2001-11-17 00:00:00.000000,17,11,November,4,2001
6,7,2001-12-28 00:00:00.000000,28,12,December,4,2001
7,8,2002-06-25 00:00:00.000000,25,6,June,2,2002
8,9,2002-08-30 00:00:00.000000,30,8,August,3,2002
9,10,2003-03-10 00:00:00.000000,10,3,March,1,2003


In [101]:
nav_history

,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639
...,...,...,...
64315,149324,2026-05-25,292.4810
64316,149324,2026-05-26,291.2707
64317,149324,2026-05-27,288.8007
64318,149324,2026-05-28,280.6873


In [102]:
fund_lookup = pd.read_sql(
    """
    SELECT fund_id, amfi_code
    FROM dim_fund
    """,
    engine
)

In [104]:
date_lookup = pd.read_sql(
    """
    SELECT date_id, full_date
    FROM dim_date
    """,
    engine
)

In [110]:
fact_nav = nav_history.merge(
    fund_lookup,
    on="amfi_code",
    how="inner"
)

In [111]:
fact_nav

,amfi_code,date,nav,fund_id
0,100016,2022-01-03,520.4608,6
1,100016,2022-01-04,515.0971,6
2,100016,2022-01-05,521.7239,6
3,100016,2022-01-06,515.7880,6
4,100016,2022-01-07,515.1639,6
...,...,...,...,...
64315,149324,2026-05-25,292.4810,40
64316,149324,2026-05-26,291.2707,40
64317,149324,2026-05-27,288.8007,40
64318,149324,2026-05-28,280.6873,40


In [112]:
nav_history

,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639
...,...,...,...
64315,149324,2026-05-25,292.4810
64316,149324,2026-05-26,291.2707
64317,149324,2026-05-27,288.8007
64318,149324,2026-05-28,280.6873


In [116]:
fact_nav['date'] = pd.to_datetime(fact_nav['date'])

In [118]:
fact_nav.dtypes

amfi_code             int64
date         datetime64[ns]
nav                 float64
fund_id               int64
dtype: object

In [120]:
date_lookup['full_date'] = pd.to_datetime(date_lookup['full_date'])

In [121]:
fact_nav = fact_nav.merge(
    date_lookup,
    left_on="date",
    right_on="full_date",
    how="inner"
)

In [122]:
fact_nav

,amfi_code,date,nav,fund_id,date_id,full_date
0,100016,2022-01-03,520.4608,6,36,2022-01-03
1,100016,2022-01-04,515.0971,6,37,2022-01-04
2,100016,2022-01-05,521.7239,6,38,2022-01-05
3,100016,2022-01-06,515.7880,6,39,2022-01-06
4,100016,2022-01-07,515.1639,6,40,2022-01-07
...,...,...,...,...,...,...
64315,149324,2026-05-25,292.4810,40,1639,2026-05-25
64316,149324,2026-05-26,291.2707,40,1640,2026-05-26
64317,149324,2026-05-27,288.8007,40,1641,2026-05-27
64318,149324,2026-05-28,280.6873,40,1642,2026-05-28


In [123]:
fact_nav = fact_nav[["fund_id","date_id","nav"]]

In [124]:
fact_nav

,fund_id,date_id,nav
0,6,36,520.4608
1,6,37,515.0971
2,6,38,521.7239
3,6,39,515.7880
4,6,40,515.1639
...,...,...,...
64315,40,1639,292.4810
64316,40,1640,291.2707
64317,40,1641,288.8007
64318,40,1642,280.6873


In [125]:
fact_nav.to_sql(
    "fact_nav",
    engine,
    if_exists="append",
    index=False
)

64320

In [126]:
pd.read_sql(
    "SELECT * FROM fact_nav LIMIT 10;",
    engine
)

,nav_id,fund_id,date_id,nav
0,1,6,36,520.4608
1,2,6,37,515.0971
2,3,6,38,521.7239
3,4,6,39,515.7880
4,5,6,40,515.1639
5,6,6,41,515.1639
6,7,6,42,515.1639
7,8,6,43,510.7136
8,9,6,44,513.5542
9,10,6,45,512.3195


In [128]:
investor_txn

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending
...,...,...,...,...,...,...,...,...,...,...,...,...,...
32773,INV003340,2025-05-30,101207,Lumpsum,168029,Madhya Pradesh,Indore,T30,26-35,Male,22.5,Net Banking,Verified
32774,INV001838,2025-05-30,119093,SIP,2175,Uttar Pradesh,Kanpur,B30,46-55,Male,27.6,Mandate,Verified
32775,INV000074,2025-05-30,120504,SIP,25998,Rajasthan,Jaipur,T30,26-35,Female,8.4,UPI,Verified
32776,INV002929,2025-05-30,148568,SIP,459,West Bengal,Kolkata,T30,26-35,Male,13.0,Mandate,Verified


In [130]:
fact_transactions = investor_txn.merge(
    fund_lookup,
    on="amfi_code",
    how="inner"
)

In [131]:
fact_transactions

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status,fund_id
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified,25
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified,35
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified,20
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending,18
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending,27
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32773,INV003340,2025-05-30,101207,Lumpsum,168029,Madhya Pradesh,Indore,T30,26-35,Male,22.5,Net Banking,Verified,30
32774,INV001838,2025-05-30,119093,SIP,2175,Uttar Pradesh,Kanpur,B30,46-55,Male,27.6,Mandate,Verified,26
32775,INV000074,2025-05-30,120504,SIP,25998,Rajasthan,Jaipur,T30,26-35,Female,8.4,UPI,Verified,12
32776,INV002929,2025-05-30,148568,SIP,459,West Bengal,Kolkata,T30,26-35,Male,13.0,Mandate,Verified,36


In [132]:
fact_transactions['transaction_date'] = pd.to_datetime(fact_transactions['transaction_date'])

In [133]:
fact_transactions.dtypes

investor_id                   object
transaction_date      datetime64[ns]
amfi_code                      int64
transaction_type              object
amount_inr                     int64
state                         object
city                          object
city_tier                     object
age_group                     object
gender                        object
annual_income_lakh           float64
payment_mode                  object
kyc_status                    object
fund_id                        int64
dtype: object

In [134]:
fact_transactions = fact_transactions.merge(
    date_lookup,
    left_on="transaction_date",
    right_on="full_date",
    how="inner"
)

In [135]:
fact_transactions

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status,fund_id,date_id,full_date
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified,25,764,2024-01-01
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified,35,764,2024-01-01
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified,20,764,2024-01-01
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending,18,764,2024-01-01
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending,27,764,2024-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32773,INV003340,2025-05-30,101207,Lumpsum,168029,Madhya Pradesh,Indore,T30,26-35,Male,22.5,Net Banking,Verified,30,1279,2025-05-30
32774,INV001838,2025-05-30,119093,SIP,2175,Uttar Pradesh,Kanpur,B30,46-55,Male,27.6,Mandate,Verified,26,1279,2025-05-30
32775,INV000074,2025-05-30,120504,SIP,25998,Rajasthan,Jaipur,T30,26-35,Female,8.4,UPI,Verified,12,1279,2025-05-30
32776,INV002929,2025-05-30,148568,SIP,459,West Bengal,Kolkata,T30,26-35,Male,13.0,Mandate,Verified,36,1279,2025-05-30


In [136]:
fact_transactions = fact_transactions[
    [
        "fund_id",
        "date_id",
        "transaction_type",
        "amount_inr",
        "investor_id",
        "state",
        "city",
        "city_tier",
        "age_group",
        "gender",
        "annual_income_lakh",
        "payment_mode",
        "kyc_status"
    ]
]

In [137]:
fact_transactions

,fund_id,date_id,transaction_type,amount_inr,investor_id,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,25,764,SIP,1834,INV003054,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,35,764,Redemption,392882,INV002952,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,20,764,SIP,912,INV003420,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,18,764,SIP,1102,INV003436,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,27,764,Lumpsum,8682,INV004691,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending
...,...,...,...,...,...,...,...,...,...,...,...,...,...
32773,30,1279,Lumpsum,168029,INV003340,Madhya Pradesh,Indore,T30,26-35,Male,22.5,Net Banking,Verified
32774,26,1279,SIP,2175,INV001838,Uttar Pradesh,Kanpur,B30,46-55,Male,27.6,Mandate,Verified
32775,12,1279,SIP,25998,INV000074,Rajasthan,Jaipur,T30,26-35,Female,8.4,UPI,Verified
32776,36,1279,SIP,459,INV002929,West Bengal,Kolkata,T30,26-35,Male,13.0,Mandate,Verified


In [138]:
fact_transactions.to_sql(
    "fact_transactions",
    engine,
    if_exists="append",
    index=False
)

32778

In [139]:
pd.read_sql(
    "SELECT * FROM fact_transactions LIMIT 5;",
    engine
)

,transaction_id,fund_id,date_id,transaction_type,amount_inr,investor_id,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,1,25,764,SIP,1834.0,INV003054,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,2,35,764,Redemption,392882.0,INV002952,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,3,20,764,SIP,912.0,INV003420,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,4,18,764,SIP,1102.0,INV003436,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,5,27,764,Lumpsum,8682.0,INV004691,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [140]:
scheme_performance

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,...,morningstar_rating,risk_grade,amfi_code_anomaly,return_1yr_pct_anomaly,return_3yr_pct_anomaly,benchmark_3yr_pct_anomaly,beta_anomaly,sharpe_ratio_anomaly,sortino_ratio_anomaly,std_dev_ann_pct_anomaly
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,...,4,Moderate,False,False,False,False,False,False,False,False
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,...,3,Moderate,False,False,False,False,False,False,False,False
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,...,5,Very High,False,True,True,True,False,False,False,False
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,...,4,Very High,False,False,True,True,False,False,False,False
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,...,5,Low,False,False,True,True,True,True,False,True
5,100016,HDFC Top 100 Fund - Regular Plan - Growth,HDFC Mutual Fund,Large Cap,Regular,10.94,14.84,11.32,14.06,0.78,...,5,Moderate,True,False,False,False,False,False,False,False
6,125497,HDFC Top 100 Fund - Direct Plan - Growth,HDFC Mutual Fund,Large Cap,Direct,11.48,13.38,13.48,12.25,1.13,...,4,Moderate,True,False,False,False,False,False,False,False
7,100033,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,HDFC Mutual Fund,Mid Cap,Regular,15.43,16.58,17.69,15.63,0.95,...,5,High,True,False,False,False,False,False,False,False
8,125498,HDFC Mid-Cap Opportunities Fund - Direct - Growth,HDFC Mutual Fund,Mid Cap,Direct,19.98,15.29,15.85,14.39,0.90,...,4,High,True,False,False,False,False,False,False,False
9,100025,HDFC Short Term Debt Fund - Regular - Growth,HDFC Mutual Fund,Short Duration,Regular,6.83,7.37,6.41,5.39,1.98,...,3,Low,True,False,False,False,True,True,True,True


In [141]:
fact_performance = scheme_performance.merge(
    fund_lookup,
    on="amfi_code",
    how="inner"
)

In [142]:
fact_performance

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,...,risk_grade,amfi_code_anomaly,return_1yr_pct_anomaly,return_3yr_pct_anomaly,benchmark_3yr_pct_anomaly,beta_anomaly,sharpe_ratio_anomaly,sortino_ratio_anomaly,std_dev_ann_pct_anomaly,fund_id
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,...,Moderate,False,False,False,False,False,False,False,False,1
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,...,Moderate,False,False,False,False,False,False,False,False,2
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,...,Very High,False,True,True,True,False,False,False,False,3
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,...,Very High,False,False,True,True,False,False,False,False,4
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,...,Low,False,False,True,True,True,True,False,True,5
5,100016,HDFC Top 100 Fund - Regular Plan - Growth,HDFC Mutual Fund,Large Cap,Regular,10.94,14.84,11.32,14.06,0.78,...,Moderate,True,False,False,False,False,False,False,False,6
6,125497,HDFC Top 100 Fund - Direct Plan - Growth,HDFC Mutual Fund,Large Cap,Direct,11.48,13.38,13.48,12.25,1.13,...,Moderate,True,False,False,False,False,False,False,False,7
7,100033,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,HDFC Mutual Fund,Mid Cap,Regular,15.43,16.58,17.69,15.63,0.95,...,High,True,False,False,False,False,False,False,False,8
8,125498,HDFC Mid-Cap Opportunities Fund - Direct - Growth,HDFC Mutual Fund,Mid Cap,Direct,19.98,15.29,15.85,14.39,0.90,...,High,True,False,False,False,False,False,False,False,9
9,100025,HDFC Short Term Debt Fund - Regular - Growth,HDFC Mutual Fund,Short Duration,Regular,6.83,7.37,6.41,5.39,1.98,...,Low,True,False,False,False,True,True,True,True,10


In [143]:
fact_performance = fact_performance[
    [
        "fund_id",
        "return_1yr_pct",
        "return_3yr_pct",
        "return_5yr_pct",
        "benchmark_3yr_pct",
        "alpha",
        "beta",
        "sharpe_ratio",
        "sortino_ratio",
        "std_dev_ann_pct",
        "max_drawdown_pct",
        "expense_ratio_pct",
        "aum_crore",
        "morningstar_rating",
        "risk_grade"
    ]
]

In [144]:
fact_performance

,fund_id,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,expense_ratio_pct,aum_crore,morningstar_rating,risk_grade
0,1,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,1.54,14288,4,Moderate
1,2,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,0.66,1231,3,Moderate
2,3,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,1.43,19259,5,Very High
3,4,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,0.72,36061,4,Very High
4,5,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,0.77,24101,5,Low
5,6,10.94,14.84,11.32,14.06,0.78,0.97,1.06,1.70,14.0,-17.41,1.55,6434,5,Moderate
6,7,11.48,13.38,13.48,12.25,1.13,0.97,0.96,1.45,14.0,-33.50,0.92,10611,4,Moderate
7,8,15.43,16.58,17.69,15.63,0.95,0.91,0.87,1.44,19.0,-13.67,1.38,23185,5,High
8,9,19.98,15.29,15.85,14.39,0.90,1.04,0.80,1.38,19.0,-32.22,0.78,18792,4,High
9,10,6.83,7.37,6.41,5.39,1.98,0.44,1.84,2.79,4.0,-6.01,0.56,27953,3,Low


In [145]:
fact_performance.to_sql(
    "fact_performance",
    engine,
    if_exists="append",
    index=False
)

40

In [146]:
pd.read_sql(
    "SELECT * FROM fact_performance LIMIT 5;",
    engine
)

,performance_id,fund_id,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,expense_ratio_pct,aum_crore,morningstar_rating,risk_grade
0,1,1,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,1.54,14288.0,4,Moderate
1,2,2,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,0.66,1231.0,3,Moderate
2,3,3,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,1.43,19259.0,5,Very High
3,4,4,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,0.72,36061.0,4,Very High
4,5,5,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,0.77,24101.0,5,Low


In [147]:
aum

,date,fund_house,aum_lakh_crore,aum_crore,num_schemes,aum_lakh_crore_anomaly,aum_crore_anomaly
0,2022-03-31,SBI Mutual Fund,6.05,605000,186,False,False
1,2022-03-31,ICICI Prudential MF,4.65,465000,216,False,False
2,2022-03-31,HDFC Mutual Fund,4.35,435000,195,False,False
3,2022-03-31,Nippon India MF,2.70,270000,177,False,False
4,2022-03-31,Kotak Mahindra MF,2.70,270000,168,False,False
...,...,...,...,...,...,...,...
85,2025-12-31,Aditya Birla Sun Life MF,4.60,460000,199,False,False
86,2025-12-31,Axis Mutual Fund,3.50,350000,95,False,False
87,2025-12-31,UTI Mutual Fund,4.10,410000,142,False,False
88,2025-12-31,Mirae Asset MF,2.90,290000,56,False,False


In [148]:
aum["date"] = pd.to_datetime(aum["date"])

In [149]:
fact_aum = aum.merge(
    date_lookup,
    left_on="date",
    right_on="full_date",
    how="inner"
)

In [150]:
fact_aum

,date,fund_house,aum_lakh_crore,aum_crore,num_schemes,aum_lakh_crore_anomaly,aum_crore_anomaly,date_id,full_date
0,2022-03-31,SBI Mutual Fund,6.05,605000,186,False,False,123,2022-03-31
1,2022-03-31,ICICI Prudential MF,4.65,465000,216,False,False,123,2022-03-31
2,2022-03-31,HDFC Mutual Fund,4.35,435000,195,False,False,123,2022-03-31
3,2022-03-31,Nippon India MF,2.70,270000,177,False,False,123,2022-03-31
4,2022-03-31,Kotak Mahindra MF,2.70,270000,168,False,False,123,2022-03-31
...,...,...,...,...,...,...,...,...,...
85,2025-12-31,Aditya Birla Sun Life MF,4.60,460000,199,False,False,1494,2025-12-31
86,2025-12-31,Axis Mutual Fund,3.50,350000,95,False,False,1494,2025-12-31
87,2025-12-31,UTI Mutual Fund,4.10,410000,142,False,False,1494,2025-12-31
88,2025-12-31,Mirae Asset MF,2.90,290000,56,False,False,1494,2025-12-31


In [151]:
fact_aum = fact_aum[
    [
        "date_id",
        "fund_house",
        "aum_lakh_crore",
        "aum_crore",
        "num_schemes"
    ]
]

In [152]:
fact_aum

,date_id,fund_house,aum_lakh_crore,aum_crore,num_schemes
0,123,SBI Mutual Fund,6.05,605000,186
1,123,ICICI Prudential MF,4.65,465000,216
2,123,HDFC Mutual Fund,4.35,435000,195
3,123,Nippon India MF,2.70,270000,177
4,123,Kotak Mahindra MF,2.70,270000,168
...,...,...,...,...,...
85,1494,Aditya Birla Sun Life MF,4.60,460000,199
86,1494,Axis Mutual Fund,3.50,350000,95
87,1494,UTI Mutual Fund,4.10,410000,142
88,1494,Mirae Asset MF,2.90,290000,56


In [153]:
fact_aum.to_sql(
    "fact_aum",
    engine,
    if_exists="append",
    index=False
)

90

In [154]:
pd.read_sql(
    "SELECT * FROM fact_aum LIMIT 5;",
    engine
)

,aum_id,date_id,fund_house,aum_lakh_crore,aum_crore,num_schemes
0,1,123,SBI Mutual Fund,6.05,605000.0,186
1,2,123,ICICI Prudential MF,4.65,465000.0,216
2,3,123,HDFC Mutual Fund,4.35,435000.0,195
3,4,123,Nippon India MF,2.70,270000.0,177
4,5,123,Kotak Mahindra MF,2.70,270000.0,168


In [155]:
a = """
SELECT
    f.scheme_name,
    d.full_date,
    n.nav
FROM fact_nav n
JOIN dim_fund f
ON n.fund_id = f.fund_id
JOIN dim_date d
ON n.date_id = d.date_id
LIMIT 10;
"""

In [165]:
import sqlite3

conn = sqlite3.connect("../bluestock_mf.db")
cursor = conn.cursor()

a = """
SELECT *
FROM dim_fund
LIMIT 5;
"""

cursor.execute(a)

print(cursor.fetchall())

conn.close()

[(1, 119551, 'SBI Mutual Fund', 'SBI Bluechip Fund - Regular Plan - Growth', 'Equity', 'Large Cap', 'Regular', '2006-02-14 00:00:00.000000', 'NIFTY 100 TRI', 'Sohini Andani', 'Moderate', 'EC01'), (2, 119552, 'SBI Mutual Fund', 'SBI Bluechip Fund - Direct Plan - Growth', 'Equity', 'Large Cap', 'Direct', '2013-01-01 00:00:00.000000', 'NIFTY 100 TRI', 'Sohini Andani', 'Moderate', 'EC01'), (3, 119598, 'SBI Mutual Fund', 'SBI Small Cap Fund - Regular Plan - Growth', 'Equity', 'Small Cap', 'Regular', '2009-09-09 00:00:00.000000', 'BSE 250 SmallCap TRI', 'R. Srinivasan', 'Very High', 'EC03'), (4, 119599, 'SBI Mutual Fund', 'SBI Small Cap Fund - Direct Plan - Growth', 'Equity', 'Small Cap', 'Direct', '2013-01-01 00:00:00.000000', 'BSE 250 SmallCap TRI', 'R. Srinivasan', 'Very High', 'EC03'), (5, 119120, 'SBI Mutual Fund', 'SBI Magnum Gilt Fund - Regular Plan - Growth', 'Debt', 'Gilt', 'Regular', '2000-12-30 00:00:00.000000', 'CRISIL Dynamic Gilt Index', 'Dinesh Ahuja', 'Low', 'DC02')]
